In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load datasets
app = pd.read_csv("application_record.csv")
credit = pd.read_csv("credit_record.csv")

In [3]:
print(f"Application: {app.shape}")
print(f"Credit: {credit.shape}")

Application: (438557, 18)
Credit: (1048575, 3)


# Target Creation

In [4]:
# Define bad client: anyone who ever had STATUS 2, 3, 4 or 5 (30+ days overdue)
credit['BAD'] = credit['STATUS'].isin(['2', '3', '4', '5']).astype(int)

In [5]:
# Aggregate by ID - if any month is BAD, client is BAD
target = credit.groupby('ID')['BAD'].max().reset_index()
target.columns = ['ID', 'TARGET']

print(f"Total unique clients in credit record: {len(target)}")
print(f"\nTarget distribution:")
print(target['TARGET'].value_counts())
print(f"\nBad client rate: {target['TARGET'].mean() * 100:.2f}%")

Total unique clients in credit record: 45985

Target distribution:
TARGET
0    45318
1      667
Name: count, dtype: int64

Bad client rate: 1.45%


In [6]:
# Merge application record with target
df = app.merge(target, on='ID', how='inner')

print(f"Application records: {len(app)}")
print(f"After merge: {len(df)}")
print(f"Lost records: {len(app) - len(df)}")
print(f"\nTarget distribution after merge:")
print(df['TARGET'].value_counts())
print(f"\nBad client rate: {df['TARGET'].mean() * 100:.2f}%")

Application records: 438557
After merge: 36457
Lost records: 402100

Target distribution after merge:
TARGET
0    35841
1      616
Name: count, dtype: int64

Bad client rate: 1.69%


In [7]:
# After inner join: 36,457 clients (36k from 438k - only clients with credit history)
# Bad clients: 616 (1.69%) - extreme class imbalance
# SMOTE will be applied before modeling

In [8]:
# 1. Drop FLAG_MOBIL - zero variance
df.drop(columns=['FLAG_MOBIL'], inplace=True)

# 2. Convert DAYS_BIRTH to AGE
df['AGE'] = (-df['DAYS_BIRTH'] / 365).astype(int)
df.drop(columns=['DAYS_BIRTH'], inplace=True)

# 3. Handle DAYS_EMPLOYED anomaly
df['IS_UNEMPLOYED'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['YEARS_EMPLOYED'] = (-df['DAYS_EMPLOYED'] / 365).round(1)
df.drop(columns=['DAYS_EMPLOYED'], inplace=True)

print(df[['AGE', 'IS_UNEMPLOYED', 'YEARS_EMPLOYED']].describe())

                AGE  IS_UNEMPLOYED  YEARS_EMPLOYED
count  36457.000000    36457.00000    30322.000000
mean      43.260334        0.16828        7.247602
std       11.510414        0.37412        6.458601
min       20.000000        0.00000        0.000000
25%       34.000000        0.00000        2.700000
50%       42.000000        0.00000        5.500000
75%       53.000000        0.00000        9.600000
max       68.000000        1.00000       43.000000


In [9]:
# 4. Binary encoding (Y/N → 1/0)
df['CODE_GENDER'] = (df['CODE_GENDER'] == 'M').astype(int)
df['FLAG_OWN_CAR'] = (df['FLAG_OWN_CAR'] == 'Y').astype(int)
df['FLAG_OWN_REALTY'] = (df['FLAG_OWN_REALTY'] == 'Y').astype(int)

# 5. Handle OCCUPATION_TYPE nulls → 'Unknown'
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

# 6. Handle YEARS_EMPLOYED nulls → 0 (they are unemployed)
df['YEARS_EMPLOYED'] = df['YEARS_EMPLOYED'].fillna(0)

# 7. Drop ID (not a feature)
df.drop(columns=['ID'], inplace=True)

# Check nulls
print("Remaining nulls:")
print(df.isnull().sum())
print(f"\nShape: {df.shape}")

Remaining nulls:
CODE_GENDER            0
FLAG_OWN_CAR           0
FLAG_OWN_REALTY        0
CNT_CHILDREN           0
AMT_INCOME_TOTAL       0
NAME_INCOME_TYPE       0
NAME_EDUCATION_TYPE    0
NAME_FAMILY_STATUS     0
NAME_HOUSING_TYPE      0
FLAG_WORK_PHONE        0
FLAG_PHONE             0
FLAG_EMAIL             0
OCCUPATION_TYPE        0
CNT_FAM_MEMBERS        0
TARGET                 0
AGE                    0
IS_UNEMPLOYED          0
YEARS_EMPLOYED         0
dtype: int64

Shape: (36457, 18)


In [10]:
# 8. One-hot encoding for categorical variables
categorical_cols = ['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 
                    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 
                    'OCCUPATION_TYPE']

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Shape after encoding: (36457, 48)

Columns: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'TARGET', 'AGE', 'IS_UNEMPLOYED', 'YEARS_EMPLOYED', 'NAME_INCOME_TYPE_Pensioner', 'NAME_INCOME_TYPE_State servant', 'NAME_INCOME_TYPE_Student', 'NAME_INCOME_TYPE_Working', 'NAME_EDUCATION_TYPE_Higher education', 'NAME_EDUCATION_TYPE_Incomplete higher', 'NAME_EDUCATION_TYPE_Lower secondary', 'NAME_EDUCATION_TYPE_Secondary / secondary special', 'NAME_FAMILY_STATUS_Married', 'NAME_FAMILY_STATUS_Separated', 'NAME_FAMILY_STATUS_Single / not married', 'NAME_FAMILY_STATUS_Widow', 'NAME_HOUSING_TYPE_House / apartment', 'NAME_HOUSING_TYPE_Municipal apartment', 'NAME_HOUSING_TYPE_Office apartment', 'NAME_HOUSING_TYPE_Rented apartment', 'NAME_HOUSING_TYPE_With parents', 'OCCUPATION_TYPE_Cleaning staff', 'OCCUPATION_TYPE_Cooking staff', 'OCCUPATION_TYPE_Core staff', 'OCCUPATION_TYPE_Drivers', 'OCCUPATION

In [12]:
# Log transformation of AMT_INCOME_TOTAL
# Reduces impact of extreme outliers (430 people earning >1M)
df['AMT_INCOME_TOTAL'] = np.log1p(df['AMT_INCOME_TOTAL'])

print("After log transformation:")
print(df['AMT_INCOME_TOTAL'].describe())

After log transformation:
count    36457.000000
mean        12.018808
std          0.480501
min         10.203629
25%         11.707678
50%         11.967187
75%         12.323860
max         14.269766
Name: AMT_INCOME_TOTAL, dtype: float64


In [13]:
# Merge Students into Working (only 17 cases, statistically irrelevant)
# In the one-hot encoded df, NAME_INCOME_TYPE_Student exists as a column
# Drop it - those 17 students will be represented as 0 in all income type columns (= Working, the base)
df.drop(columns=['NAME_INCOME_TYPE_Student'], inplace=True)

print(f"Shape after dropping Student column: {df.shape}")

Shape after dropping Student column: (36457, 47)


In [14]:
# NOTE: Feature scaling (StandardScaler) will be applied in modeling notebook
# Must be fit ONLY on training data to avoid data leakage
# Then transform both train and test sets separately

In [15]:
# Save processed dataset
df.to_csv("data_processed.csv", index=False)
print("Dataset saved: data_processed.csv")
print(f"Final shape: {df.shape}")

Dataset saved: data_processed.csv
Final shape: (36457, 47)
